# PPO after CoT SFT (Reasoning Training Data)

Second-stage **RL fine-tuning** on top of `r1_distill_cot_sft.ipynb`, using **verifiable rewards** (exact answer match).

## Is your actor/critic picture correct?

| Component | Common misconception | Standard PPO (what we implement) |
|-----------|---------------------|----------------------------------|
| **Actor (policy)** | Generates *only* reasoning | Generates the **full continuation**: `\n` + reasoning + `\n` + answer |
| **Critic (value)** | Predicts error vs ground-truth answer | Predicts **expected future return** \(V(s)\) from the current token prefix (used for GAE advantages) |
| **Reward** | (sometimes folded into critic) | **Separate signal** at end of rollout: e.g. +1 if parsed answer matches GT, else 0 |

A model that directly predicts "difference from GT answer" is closer to a **reward model** or **regression critic**, not the usual PPO value function. For this dataset, a **rule-based terminal reward** (exact match) is simpler and matches the original GRPO-style design.

**Optional variant (not default):** actor generates reasoning only, then a fixed decoder step produces the answer — still use rule-based reward on the answer; value head remains a return predictor, not an answer-diff head.

In [ ]:
import importlib.util
import os
import subprocess
import sys

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
IN_COLAB = "google.colab" in sys.modules

REQUIRED = [
    "torch", "transformers", "datasets", "trl", "peft", "accelerate",
    "pandas", "matplotlib", "tqdm",
]
if IN_COLAB or sys.platform != "win32":
    REQUIRED.append("bitsandbytes")

missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Runtime:", "Colab" if IN_COLAB else "Local", "|", sys.executable)

In [ ]:
import json
import random
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import Dataset
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoTokenizer, BitsAndBytesConfig
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer

IN_COLAB = "google.colab" in sys.modules
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

SMOKE_TEST = False if (IN_COLAB and torch.cuda.is_available()) else not torch.cuda.is_available()

MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
THINK_START = "\u003cthink\u003e"
THINK_END = "\u003c/think\u003e"
MAX_PROMPT_LENGTH = 512
MAX_RESPONSE_LENGTH = 384


def resolve_paths():
    roots = [
        Path(".."),
        Path("/content/reasoning"),
        Path("."),
    ]
    for root in roots:
        data = root / "reasoning_training_dataset" / "1B_Model_Low_Reasoning_Data.csv"
        sft = root / "outputs" / "r1_distill_cot_sft" / "final_lora"
        if data.exists():
            out = root / "outputs" / "r1_distill_ppo"
            return data.resolve(), (sft.resolve() if sft.exists() else None), out.resolve()
    raise FileNotFoundError("Could not find reasoning_training_dataset/1B_Model_Low_Reasoning_Data.csv")


DATA_PATH, SFT_ADAPTER_PATH, OUTPUT_DIR = resolve_paths()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| SMOKE_TEST:", SMOKE_TEST)
print("DATA:", DATA_PATH)
print("SFT adapter:", SFT_ADAPTER_PATH or "(none — using base model)")
print("OUT:", OUTPUT_DIR)

## 1. Data & prompts

PPO needs **prompts** + **ground-truth answers** (for reward only). Reasoning is *not* shown to the reward — only the parsed final answer matters.

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["prompt", "answer"])
df["prompt"] = df["prompt"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()
df = df.drop_duplicates(subset=["prompt", "answer"]).reset_index(drop=True)

if SMOKE_TEST:
    df = df.sample(min(64, len(df)), random_state=SEED).reset_index(drop=True)

train_df = df.sample(frac=0.9, random_state=SEED)
test_df = df.drop(train_df.index)
print(f"PPO train prompts: {len(train_df)} | holdout: {len(test_df)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # important for batched generation in PPO


def build_messages(prompt: str) -> list[dict]:
    return [{"role": "user", "content": prompt}]


def build_query_text(prompt: str) -> str:
    try:
        text = tokenizer.apply_chat_template(
            build_messages(prompt), tokenize=False, add_generation_prompt=True, enable_thinking=True
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            build_messages(prompt), tokenize=False, add_generation_prompt=True
        )
    if THINK_START not in text:
        text += f"{THINK_START}\n"
    return text


def extract_answer(generated: str) -> str:
    if THINK_END in generated:
        tail = generated.split(THINK_END)[-1]
    else:
        tail = generated
    for line in tail.splitlines():
        line = line.strip()
        if line:
            return line
    return tail.strip()


def normalize_answer(text: str) -> str:
    text = text.strip().strip('"').strip("'")
    return re.sub(r"\s+", " ", text).lower()


def compute_reward(generated: str, gold: str) -> float:
    """Terminal reward for PPO (not the critic)."""
    pred = extract_answer(generated)
    r = 1.0 if normalize_answer(pred) == normalize_answer(gold) else 0.0
    if THINK_END in generated:
        r += 0.05  # small format bonus
    return r


example = train_df.iloc[0]
print("Query prefix (actor conditions on this):")
print(build_query_text(example["prompt"])[:500], "...")
print("Gold answer (reward only):", example["answer"])

## 2. Models: policy (+ value head), reference

- **Policy** = actor: `AutoModelForCausalLMWithValueHead` (+ optional SFT LoRA)
- **Value head** = critic in TRL/PPO: scalar \(V(s)\) per token for advantage estimation
- **Ref model** = frozen snapshot for KL penalty

In [ ]:
USE_4BIT = device == "cuda"
try:
    import bitsandbytes  # noqa: F401
except ImportError:
    USE_4BIT = False

quant_config = None
if USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

common_kwargs = dict(trust_remote_code=True, device_map="auto" if device == "cuda" else None)
if quant_config:
    common_kwargs["quantization_config"] = quant_config

policy = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_ID, **common_kwargs)
ref_policy = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_ID, **common_kwargs)

if SFT_ADAPTER_PATH is not None:
    policy.pretrained_model = PeftModel.from_pretrained(
        policy.pretrained_model, str(SFT_ADAPTER_PATH), is_trainable=True
    )
    ref_policy.pretrained_model = PeftModel.from_pretrained(
        ref_policy.pretrained_model, str(SFT_ADAPTER_PATH), is_trainable=False
    )

ref_policy.eval()
for p in ref_policy.parameters():
    p.requires_grad = False

print("Policy + value head ready; ref model frozen.")

In [ ]:
ppo_config = PPOConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    learning_rate=1e-5,
    batch_size=4 if not SMOKE_TEST else 2,
    mini_batch_size=2 if not SMOKE_TEST else 1,
    gradient_accumulation_steps=1,
    ppo_epochs=2,
    kl_coef=0.05,
    gamma=1.0,
    lam=0.95,
    cliprange=0.2,
    vf_coef=0.1,
    max_grad_norm=1.0,
    logging_steps=1,
    response_length=MAX_RESPONSE_LENGTH,
    report_to="none",
    seed=SEED,
)

try:
    ppo_trainer = PPOTrainer(
        args=ppo_config,
        processing_class=tokenizer,
        model=policy,
        ref_model=ref_policy,
    )
except TypeError:
    ppo_trainer = PPOTrainer(
        config=ppo_config,
        model=policy,
        ref_model=ref_policy,
        tokenizer=tokenizer,
    )

generation_kwargs = dict(
    max_new_tokens=MAX_RESPONSE_LENGTH,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
print(ppo_config)

## 3. PPO training loop

For each batch:
1. **Actor** rolls out responses from query prefixes
2. **Reward fn** scores terminal answer vs GT
3. **PPO step** updates policy + **value head** (critic) using advantages

In [ ]:
def tokenize_query(prompt: str) -> torch.Tensor:
    q = build_query_text(prompt)
    ids = tokenizer(q, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH)["input_ids"][0]
    return ids


NUM_PPO_STEPS = 3 if SMOKE_TEST else 50
BATCH_PROMPTS = 4 if not SMOKE_TEST else 2

train_prompts = train_df["prompt"].tolist()
train_golds = train_df["answer"].tolist()

history = []
rng = random.Random(SEED)

for step in tqdm(range(NUM_PPO_STEPS), desc="PPO steps"):
    idxs = [rng.randrange(len(train_prompts)) for _ in range(BATCH_PROMPTS)]
    batch_prompts = [train_prompts[i] for i in idxs]
    batch_golds = [train_golds[i] for i in idxs]

    query_tensors = [tokenize_query(p) for p in batch_prompts]

    response_tensors = ppo_trainer.generate(query_tensors, return_prompt=False, **generation_kwargs)

    decoded = [tokenizer.decode(r, skip_special_tokens=True) for r in response_tensors]
    rewards = [torch.tensor(compute_reward(gen, gold)) for gen, gold in zip(decoded, batch_golds)]

    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
    stats = {k: float(v) if torch.is_tensor(v) else v for k, v in stats.items()}
    stats["mean_reward"] = float(torch.stack(rewards).mean())
    stats["step"] = step
    history.append(stats)

hist_df = pd.DataFrame(history)
hist_df.to_csv(OUTPUT_DIR / "ppo_history.csv", index=False)
hist_df.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
if "ppo/loss/total" in hist_df.columns:
    axes[0].plot(hist_df["step"], hist_df["ppo/loss/total"], marker="o")
    axes[0].set_title("PPO total loss")
elif "loss" in hist_df.columns:
    axes[0].plot(hist_df["step"], hist_df["loss"], marker="o")
    axes[0].set_title("PPO loss")
axes[1].plot(hist_df["step"], hist_df["mean_reward"], marker="o", color="green")
axes[1].set_title("Mean terminal reward")
for ax in axes:
    ax.set_xlabel("PPO step")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ppo_curves.png", dpi=150)
plt.show()

In [ ]:
save_dir = OUTPUT_DIR / "final_ppo"
save_dir.mkdir(exist_ok=True)
policy.save_pretrained(str(save_dir))
tokenizer.save_pretrained(str(save_dir))
print("Saved to", save_dir)

## 4. Evaluate on holdout (same metric as SFT)

In [ ]:
eval_n = min(32 if SMOKE_TEST else len(test_df), len(test_df))
eval_df = test_df.sample(eval_n, random_state=SEED)

preds, golds = [], []
policy.eval()
for prompt, gold in tqdm(list(zip(eval_df["prompt"], eval_df["answer"])), desc="Eval"):
    q = tokenize_query(prompt).unsqueeze(0).to(policy.pretrained_model.device)
    with torch.no_grad():
        out = policy.generate(q, **{**generation_kwargs, "do_sample": False, "max_new_tokens": MAX_RESPONSE_LENGTH})
    gen = tokenizer.decode(out[0][q.shape[1]:], skip_special_tokens=True)
    preds.append(extract_answer(gen))
    golds.append(gold)

correct = [normalize_answer(p) == normalize_answer(g) for p, g in zip(preds, golds)]
acc = sum(correct) / len(correct)
print(f"Holdout exact-match accuracy: {acc:.2%} ({sum(correct)}/{len(correct)})")

summary = {
    "stage": "ppo",
    "model_id": MODEL_ID,
    "sft_adapter": str(SFT_ADAPTER_PATH) if SFT_ADAPTER_PATH else None,
    "ppo_steps": NUM_PPO_STEPS,
    "holdout_accuracy": acc,
    "reward": "terminal exact match (+ small format bonus)",
    "critic": "AutoModelForCausalLMWithValueHead (standard PPO value function)",
}
with open(OUTPUT_DIR / "eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))